In [ ]:
"""
Energy visualization options:
1. Energy consumption by transit line
2. Distribution of energy across the fleet
3. Energy vs. travel time by bus
"""

import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

In [ ]:
# Load the data
df = pd.read_csv("transit_departure_updated_energy.csv", index_col=0)

target_lines = ['1-1','1-3','1-4','1-9','1-11','1-13','1-16','1-18','1-19','1-22','1-25','1-28','1-29','1-32','1-33','1-36','1-39','1-51','1-52','1-53','1-54','1-55','1-56','1-57','1-58','1-59','1-61','1-64','1-65','1-70','1-72','1-74','1-76','1-79','1-80','1-81','1-82','1-84','1-85','1-86','1-88','1-90','1-91','1-92','1-93','1-94','1-96','1-107','1-133','1-136','1-185','1-400','1-401']
df = df[df["Line ID"].isin(target_lines)].copy()
# Apply correction factor to energy consumption
correction_factor = 1.28
df['Energy Consumption (kWh)'] = df['Energy Consumption (kWh)'] * correction_factor

# Convert to timedelta to handle times beyond 24:00
df['Departure Time'] = pd.to_timedelta(df['Departure Time'])
df['Time (hours)'] = df['Departure Time'].dt.total_seconds() / 3600

print(f"Total records: {len(df)}")
print(f"Total unique buses: {df['Vehicle ID'].nunique()}")
print(f"Total unique lines: {df['Line ID'].nunique()}")
print(f"Date range (if applicable): {df['Departure Time'].min()} to {df['Departure Time'].max()}")

In [ ]:
# Set style to match matsim_data_visualization.py
plt.rcParams.update({
    "font.size": 14,
    "axes.titlesize": 18,
    "axes.labelsize": 16,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "legend.fontsize": 13
})
sns.set_style("whitegrid")

# Create figure with 4 subplots
fig, axes = plt.subplots(2, 2, figsize=(16, 11))

# ========== Plot 1: Energy by Line (box plot) ==========
ax1 = axes[0, 0]
line_ids = sorted(df['Line ID'].unique())
energy_by_line = [df[df['Line ID'] == line]['Energy Consumption (kWh)'].values for line in line_ids]

bp = ax1.boxplot(energy_by_line, labels=line_ids, patch_artist=True)
for patch in bp['boxes']:
    patch.set_facecolor('#4C72B0')
    patch.set_alpha(0.7)
ax1.set_xlabel('Transit Line')
ax1.set_ylabel('Total Consumption (kWh)')
ax1.set_title('(a) Energy Distribution by Transit Line')
ax1.grid(axis='y', linestyle='--', alpha=0.7)
# Rotate labels to vertical for better readability with many lines
ax1.tick_params(axis='x', rotation=90)

# Add mean markers
means = [np.mean(data) for data in energy_by_line]
ax1.scatter(range(1, len(means)+1), means, color='#E15759', s=100, zorder=3, marker='D', label='Mean')
ax1.legend()

# ========== Plot 2: Energy Consumption vs Travel Time ==========
ax2 = axes[0, 1]
scatter = ax2.scatter(
    df['Travel Time (min)'], 
    df['Energy Consumption (kWh)'],
    c=df['Time (hours)'],
    cmap='viridis',
    s=80,
    alpha=0.7,
    edgecolor='k',
    linewidth=0.5
)
ax2.set_xlabel('Travel Time (min)')
ax2.set_ylabel('Energy Consumption (kWh)')
ax2.set_title('(b) Energy vs Travel Time (colored by time of day)')
ax2.grid(True, linestyle='--', alpha=0.7)
cbar = plt.colorbar(scatter, ax=ax2)
cbar.set_label('Time of Day (hours)')

# Add trend line
z = np.polyfit(df['Travel Time (min)'], df['Energy Consumption (kWh)'], 1)
p = np.poly1d(z)
x_trend = np.linspace(df['Travel Time (min)'].min(), df['Travel Time (min)'].max(), 100)
ax2.plot(x_trend, p(x_trend), "r--", linewidth=2, label=f'Trend: y={z[0]:.3f}x+{z[1]:.3f}')
ax2.legend()

# ========== Plot 3: Histogram of Energy Distribution ==========
ax3 = axes[1, 0]
ax3.hist(df['Energy Consumption (kWh)'], bins=30, color='#4C72B0', edgecolor='#2E4A7C', alpha=0.7)
ax3.axvline(df['Energy Consumption (kWh)'].mean(), color='#E15759', linestyle='--', linewidth=2, label=f"Mean: {df['Energy Consumption (kWh)'].mean():.2f} kWh")
ax3.axvline(df['Energy Consumption (kWh)'].median(), color='#FF9D00', linestyle='--', linewidth=2, label=f"Median: {df['Energy Consumption (kWh)'].median():.2f} kWh")
ax3.set_xlabel('Energy Consumption (kWh)')
ax3.set_ylabel('Number of Trips')
ax3.set_title('(c) Distribution of Energy Consumption')
ax3.legend()
ax3.grid(axis='y', linestyle='--', alpha=0.7)

# ========== Plot 4: Energy Efficiency (kWh/km) by Line ==========
ax4 = axes[1, 1]
df['Energy per km'] = df['Energy Consumption (kWh)'] / (df['Distance (m)'] / 1000)

efficiency_by_line = df.groupby('Line ID')['Energy per km'].agg(['mean', 'std']).reset_index()
if len(efficiency_by_line) > 0:
    ax4.bar(
        range(len(efficiency_by_line)), 
        efficiency_by_line['mean'],
        yerr=efficiency_by_line['std'],
        capsize=5,
        color='#55A868',
        edgecolor='#3A7D4A',
        alpha=0.7
    )
    ax4.set_xticks(range(len(efficiency_by_line)))
    ax4.set_xticklabels(efficiency_by_line['Line ID'], rotation=90)
    ax4.set_xlabel('Transit Line')
    ax4.set_ylabel('Energy Consumption (kWh/km)')
    ax4.set_title('(d) Mean Energy Consumption by Line')
    ax4.grid(axis='y', linestyle='--', alpha=0.7)
    
    # Add mean marker
    overall_mean = 1.19
    ax4.axhline(overall_mean, color='#E15759', linestyle='--', linewidth=2, label=f'Fleet Mean: {overall_mean:.2f} kWh/km')
    ax4.legend()

plt.tight_layout()
plt.show()

In [ ]:
# Print detailed statistics
print("\n" + "="*70)
print("SUMMARY STATISTICS")
print("="*70)
print(f"\nTotal Energy Consumption: {df['Energy Consumption (kWh)'].sum():.2f} kWh")
print(f"Mean per trip: {df['Energy Consumption (kWh)'].mean():.2f} kWh")
print(f"Median per trip: {df['Energy Consumption (kWh)'].median():.2f} kWh")
print(f"Std deviation: {df['Energy Consumption (kWh)'].std():.2f} kWh")
print(f"Min: {df['Energy Consumption (kWh)'].min():.2f} kWh | Max: {df['Energy Consumption (kWh)'].max():.2f} kWh")

print(f"\nEnergy Efficiency (kWh/km):")
print(f"  Mean: {df['Energy per km'].mean():.4f} kWh/km")
print(f"  Median: {df['Energy per km'].median():.4f} kWh/km")
print(f"  Std dev: {df['Energy per km'].std():.4f} kWh/km")

print(f"\n--- By Transit Line ---")
for line in sorted(df['Line ID'].unique()):
    line_data = df[df['Line ID'] == line]
    print(f"\nLine {line}:")
    print(f"  Trips: {len(line_data)}")
    print(f"  Total energy: {line_data['Energy Consumption (kWh)'].sum():.2f} kWh")
    print(f"  Mean energy: {line_data['Energy Consumption (kWh)'].mean():.2f} kWh")
    print(f"  Efficiency: {line_data['Energy per km'].mean():.4f} kWh/km")